# 📘 **FULL PIPELINE: GPT‑4o Vision OCR → OpenAI Embeddings → Azure ML → GPT‑4o Reasoning**
### Updated Notebook (No HuggingFace API) — Beginner Friendly
This notebook includes every step you need:
1. Load MultiFinBen-EnglishOCR dataset from Azure Storage
2. GPT‑4o Vision OCR (correct 2026 API format)
3. Budget‑safe OCR extraction
4. OpenAI embeddings (replaces HuggingFace)
5. Optional fusion model training
6. Azure ML scoring script
7. GPT‑4o reasoning summary


In [ ]:
!pip install openai pandas tqdm pyarrow scikit-learn joblib requests

In [ ]:
from openai import OpenAI
import pandas as pd, numpy as np, time, os, joblib, requests
from tqdm import tqdm

OPENAI_API_KEY = 'YOUR_OPENAI_KEY'
client = OpenAI(api_key=OPENAI_API_KEY)

## 📥 Step 1 — Load Dataset

In [ ]:
sas_url = 'https://<yourstorage>.blob.core.windows.net/multifinben/raw/train-00000-of-00008.parquet?<sas>'
df = pd.read_parquet(sas_url)
df.head()

## 👁️ Step 2 — GPT‑4o Vision OCR (Correct API)

In [ ]:
def gpt4o_vision_ocr(base64_str):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'text', 'text': 'Extract ONLY the text from this image.'},
                {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{base64_str}'}}
            ]
        }]
    )
    return response.choices[0].message.content

## 💸 Step 3 — Budget‑Safe OCR Loop

In [ ]:
MAX_SPEND_USD = 8.0
COST_PER_IMAGE = 0.005
OUTPUT = 'ocr_budget_safe.pkl'

if 'ocr_text' not in df.columns: df['ocr_text'] = None
if os.path.exists(OUTPUT): df.update(pd.read_pickle(OUTPUT))

spent = 0.0
processed = 0

for i in tqdm(range(len(df))):
    if isinstance(df.loc[i,'ocr_text'], str) and len(df.loc[i,'ocr_text'])>1: continue
    if spent + COST_PER_IMAGE > MAX_SPEND_USD: break
    try:
        text = gpt4o_vision_ocr(df.loc[i,'image'])
        df.loc[i,'ocr_text'] = text
        spent += COST_PER_IMAGE
        processed += 1
        if processed % 20 == 0: df.to_pickle(OUTPUT)
        time.sleep(0.3)
    except Exception as e:
        print('Error:', e); time.sleep(1)

df.to_pickle(OUTPUT)
print('OCR DONE:', processed, 'images | Cost ≈ $', spent)

## 🔡 Step 4 — Generate OpenAI Embeddings (No HuggingFace)

In [ ]:
def embed_text(text):
    if text is None or len(text.strip()) == 0: return None
    response = client.embeddings.create(
        model='text-embedding-3-large',
        input=text
    )
    return response.data[0].embedding

In [ ]:
df['embedding'] = [embed_text(t) for t in tqdm(df['ocr_text'])]
df.to_pickle('processed_embeddings.pkl')
df.head()

## 🤖 Step 5 — Train Fusion Model (Optional)

In [ ]:
df_model = pd.read_pickle('processed_embeddings.pkl')
X = np.vstack(df_model['embedding'])
y = df_model['label']  # if labels exist

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X,y)
joblib.dump(model,'fusion_model.joblib')
model

## ☁️ Step 6 — Azure ML Real-Time Scoring Script

In [ ]:
%%writefile score.py
import json, numpy as np, joblib, os
def init():
    global model
    model = joblib.load(os.path.join(os.environ['AZUREML_MODEL_DIR'], 'fusion_model.joblib'))
def run(raw):
    d = json.loads(raw)
    emb = np.array(d['embedding'])
    return {'score': float(model.predict_proba([emb])[0][1])}

## 📊 Step 7 — GPT‑4o Financial Reasoning & Summary

In [ ]:
def gpt4o_reason(text):
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role':'system','content':'You are a financial analyst.'},
            {'role':'user','content': text}
        ]
    )
    return response.choices[0].message.content

gpt4o_reason(df['ocr_text'].iloc[0])